<a href="https://colab.research.google.com/github/sv2001eta-svg/student_PPK/blob/main/pr_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 функции ClickHouse в образовательной аналитике

**Тема:** функции ClickHouse  
**Контекст:** образовательная аналитика LMS  
**Формат:** теория → сквозной пример → SQL-разбор → задания

## Цели занятия

После выполнения тетради студент должен уметь:

1. Извлекать части даты и времени: `toYear`, `toMonth`, `toDayOfWeek`, `toHour`.
2. Округлять временные метки для временных рядов: `toStartOfMonth`, `toStartOfDay`, `toStartOfHour`.
3. Считать интервалы между событиями через `dateDiff`.
4. Использовать агрегатные функции: `argMax`, `argMin`, `uniq`, `uniqExact`, `groupArray`.
5. Анализировать траектории студентов через массивы: `indexOf`, `arraySlice`, `has`, `hasAll`, `hasAny`.
6. Строить образовательные метрики: DAU, MAU, activation, completion, dropout-risk.

---

## Сквозной кейс

Мы анализируем события LMS-платформы:

- студент зарегистрировался;
- впервые вошёл в систему;
- открывал и завершал модули;
- отправлял тесты;
- получил сертификат или ушёл в отток.

Все примеры ниже работают с одной таблицей: `lect_04_lms_events`.

## 0. Установка и импорт библиотек

Ячейка повторяет подход из подготовительной тетради: устанавливаем `clickhouse-connect`, импортируем `pandas` и создаём удобную функцию для выполнения SQL.

In [46]:
!pip install -q clickhouse-connect pandas plotly

In [2]:
import os
import getpass
import pandas as pd
import clickhouse_connect

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)

print('Окружение готово ✔')

Окружение готово ✔


## 1. Подключение к ClickHouse

Заполните выданные в курсе реквизиты.  
В отличие от демонстрационного ноутбука, пароль не записывается в код — его безопаснее вводить через `getpass`.

> Если преподаватель проводит занятие в закрытом контуре, можно заранее заменить значения `CH_USER`, `CH_DATABASE` и `CH_PASSWORD` на учебные.

In [47]:
CH_HOST = '95.131.149.21'
CH_PORT = 8123

CH_USER = os.getenv('CH_USER') or input('ClickHouse username: ')
CH_DATABASE = os.getenv('CH_DATABASE') or input('ClickHouse database: ')
CH_PASSWORD = os.getenv('CH_PASSWORD') or getpass.getpass('ClickHouse password: ')

client = clickhouse_connect.get_client(
    host=CH_HOST,
    port=CH_PORT,
    username=CH_USER,
    password=CH_PASSWORD,
    database=CH_DATABASE,
    secure=False,
)

server_version = client.command('SELECT version()')
print(f'✔ Подключились к ClickHouse, версия сервера: {server_version}')

ClickHouse username: user_169
ClickHouse database: db_169
ClickHouse password: ··········
✔ Подключились к ClickHouse, версия сервера: 26.2.4.23


## 2. Создание учебного LMS-датасета

Сформируем небольшой журнал событий `lms_events`.

### Поля таблицы

| Поле | Смысл |
|---|---|
| `event_id` | идентификатор события |
| `student_id` | идентификатор студента |
| `course_id` | курс |
| `module_id` | модуль курса |
| `event_type` | тип события |
| `event_time` | время события |
| `score` | балл, если событие связано с оцениванием |
| `time_spent_min` | длительность активности |
| `registration_date` | дата регистрации |
| `birth_date` | дата рождения |

In [48]:
events = [
    # student 101: успешная траектория
    (1, 101, 'CH101', 'M1', 'registered',          '2025-09-01 09:00:00', None,  0, '2025-09-01', '2004-05-20'),
    (2, 101, 'CH101', 'M1', 'first_login',         '2025-09-01 10:15:00', None, 20, '2025-09-01', '2004-05-20'),
    (3, 101, 'CH101', 'M1', 'module_completed',    '2025-09-03 18:30:00', 78,  85, '2025-09-01', '2004-05-20'),
    (4, 101, 'CH101', 'M2', 'module_completed',    '2025-09-05 20:10:00', 82,  90, '2025-09-01', '2004-05-20'),
    (5, 101, 'CH101', 'M3', 'module_completed',    '2025-09-10 21:00:00', 88, 110, '2025-09-01', '2004-05-20'),
    (6, 101, 'CH101', 'FINAL', 'final_test_passed','2025-09-15 19:30:00', 91,  70, '2025-09-01', '2004-05-20'),
    (7, 101, 'CH101', 'CERT', 'certificate_issued','2025-09-16 09:00:00', None,  0, '2025-09-01', '2004-05-20'),

    # student 102: прошёл ядро, но без сертификата
    (8, 102, 'CH101', 'M1', 'registered',          '2025-09-02 12:00:00', None,  0, '2025-09-02', '2003-11-03'),
    (9, 102, 'CH101', 'M1', 'first_login',         '2025-09-04 08:20:00', None, 15, '2025-09-02', '2003-11-03'),
    (10,102, 'CH101', 'M1', 'module_completed',    '2025-09-06 17:00:00', 64,  60, '2025-09-02', '2003-11-03'),
    (11,102, 'CH101', 'M2', 'module_completed',    '2025-09-09 16:40:00', 71,  75, '2025-09-02', '2003-11-03'),
    (12,102, 'CH101', 'M3', 'module_completed',    '2025-09-15 14:00:00', 69,  80, '2025-09-02', '2003-11-03'),

    # student 103: ранний отток
    (13,103, 'CH101', 'M1', 'registered',          '2025-09-03 15:00:00', None,  0, '2025-09-03', '2005-02-17'),
    (14,103, 'CH101', 'M1', 'first_login',         '2025-09-10 15:30:00', None,  5, '2025-09-03', '2005-02-17'),
    (15,103, 'CH101', 'M1', 'warning',             '2025-09-15 12:00:00', None,  0, '2025-09-03', '2005-02-17'),
    (16,103, 'CH101', 'M1', 'dropout',             '2025-09-20 12:00:00', None,  0, '2025-09-03', '2005-02-17'),

    # student 104: поздний вечерний студент
    (17,104, 'CH101', 'M1', 'registered',          '2025-09-04 20:00:00', None,  0, '2025-09-04', '1999-07-29'),
    (18,104, 'CH101', 'M1', 'first_login',         '2025-09-04 22:10:00', None, 30, '2025-09-04', '1999-07-29'),
    (19,104, 'CH101', 'M1', 'module_completed',    '2025-09-07 23:00:00', 90, 100, '2025-09-04', '1999-07-29'),
    (20,104, 'CH101', 'M2', 'module_completed',    '2025-09-08 22:40:00', 93, 105, '2025-09-04', '1999-07-29'),
    (21,104, 'CH101', 'M3', 'module_completed',    '2025-09-11 23:20:00', 95, 120, '2025-09-04', '1999-07-29'),
    (22,104, 'CH101', 'ADV1', 'advanced_started',  '2025-09-13 22:30:00', None, 40, '2025-09-04', '1999-07-29'),
    (23,104, 'CH101', 'FINAL', 'final_test_passed','2025-09-18 21:00:00', 97,  80, '2025-09-04', '1999-07-29'),

    # student 105: активен, но слабый балл
    (24,105, 'CH101', 'M1', 'registered',          '2025-09-05 09:10:00', None,  0, '2025-09-05', '2002-12-31'),
    (25,105, 'CH101', 'M1', 'first_login',         '2025-09-05 09:40:00', None, 25, '2025-09-05', '2002-12-31'),
    (26,105, 'CH101', 'M1', 'module_completed',    '2025-09-06 11:00:00', 55, 100, '2025-09-05', '2002-12-31'),
    (27,105, 'CH101', 'M2', 'module_completed',    '2025-09-12 11:00:00', 58, 130, '2025-09-05', '2002-12-31'),
    (28,105, 'CH101', 'M2', 'warning',             '2025-09-13 12:00:00', None,  0, '2025-09-05', '2002-12-31'),

    # student 106: без первого входа
    (29,106, 'CH101', 'M1', 'registered',          '2025-09-07 10:00:00', None,  0, '2025-09-07', '2001-01-15'),

    # student 107: быстрый студент
    (30,107, 'CH101', 'M1', 'registered',          '2025-09-08 08:00:00', None,  0, '2025-09-08', '2004-09-10'),
    (31,107, 'CH101', 'M1', 'first_login',         '2025-09-08 08:05:00', None, 10, '2025-09-08', '2004-09-10'),
    (32,107, 'CH101', 'M1', 'module_completed',    '2025-09-08 09:10:00', 86,  50, '2025-09-08', '2004-09-10'),
    (33,107, 'CH101', 'M2', 'module_completed',    '2025-09-08 10:00:00', 88,  55, '2025-09-08', '2004-09-10'),
    (34,107, 'CH101', 'M3', 'module_completed',    '2025-09-09 11:00:00', 87,  60, '2025-09-08', '2004-09-10'),
    (35,107, 'CH101', 'FINAL', 'final_test_passed','2025-09-09 12:30:00', 89,  45, '2025-09-08', '2004-09-10'),

    # student 108: продвинутый трек
    (36,108, 'CH101', 'M1', 'registered',          '2025-09-10 13:00:00', None,  0, '2025-09-10', '1998-04-02'),
    (37,108, 'CH101', 'M1', 'first_login',         '2025-09-11 13:20:00', None, 35, '2025-09-10', '1998-04-02'),
    (38,108, 'CH101', 'M1', 'module_completed',    '2025-09-12 18:00:00', 77,  85, '2025-09-10', '1998-04-02'),
    (39,108, 'CH101', 'M2', 'module_completed',    '2025-09-15 18:00:00', 81,  90, '2025-09-10', '1998-04-02'),
    (40,108, 'CH101', 'ADV2', 'advanced_started',  '2025-09-16 19:00:00', None, 45, '2025-09-10', '1998-04-02'),
]

columns = [
    'event_id', 'student_id', 'course_id', 'module_id', 'event_type',
    'event_time', 'score', 'time_spent_min', 'registration_date', 'birth_date'
]

df_events = pd.DataFrame(events, columns=columns)
df_events['event_time'] = pd.to_datetime(df_events['event_time'])
df_events['registration_date'] = pd.to_datetime(df_events['registration_date']).dt.date
df_events['birth_date'] = pd.to_datetime(df_events['birth_date']).dt.date
df_events['score'] = df_events['score'].astype('float')

df_events.head(10)

,event_id,student_id,course_id,module_id,event_type,event_time,score,time_spent_min,registration_date,birth_date
0,1,101,CH101,M1,registered,2025-09-01 09:00:00,NaN,0,2025-09-01,2004-05-20
1,2,101,CH101,M1,first_login,2025-09-01 10:15:00,NaN,20,2025-09-01,2004-05-20
2,3,101,CH101,M1,module_completed,2025-09-03 18:30:00,78.0,85,2025-09-01,2004-05-20
3,4,101,CH101,M2,module_completed,2025-09-05 20:10:00,82.0,90,2025-09-01,2004-05-20
4,5,101,CH101,M3,module_completed,2025-09-10 21:00:00,88.0,110,2025-09-01,2004-05-20
5,6,101,CH101,FINAL,final_test_passed,2025-09-15 19:30:00,91.0,70,2025-09-01,2004-05-20
6,7,101,CH101,CERT,certificate_issued,2025-09-16 09:00:00,NaN,0,2025-09-01,2004-05-20
7,8,102,CH101,M1,registered,2025-09-02 12:00:00,NaN,0,2025-09-02,2003-11-03
8,9,102,CH101,M1,first_login,2025-09-04 08:20:00,NaN,15,2025-09-02,2003-11-03
9,10,102,CH101,M1,module_completed,2025-09-06 17:00:00,64.0,60,2025-09-02,2003-11-03


In [49]:
df_events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   event_id           40 non-null     int64         
 1   student_id         40 non-null     int64         
 2   course_id          40 non-null     object        
 3   module_id          40 non-null     object        
 4   event_type         40 non-null     object        
 5   event_time         40 non-null     datetime64[ns]
 6   score              19 non-null     float64       
 7   time_spent_min     40 non-null     int64         
 8   registration_date  40 non-null     object        
 9   birth_date         40 non-null     object        
dtypes: datetime64[ns](1), float64(1), int64(3), object(5)
memory usage: 3.3+ KB


## 3. Создание таблицы и загрузка данных в ClickHouse

Ячейка идемпотентна: таблица создаётся при отсутствии, затем очищается и заново загружается.

In [50]:
DDL_LMS_EVENTS = '''
CREATE TABLE IF NOT EXISTS lect_04_lms_events (
    event_id UInt32,
    student_id UInt32,
    course_id String,
    module_id String,
    event_type String,
    event_time DateTime,
    score Nullable(Float32),
    time_spent_min UInt16,
    registration_date Date,
    birth_date Date
)
ENGINE = MergeTree()
ORDER BY (course_id, student_id, event_time, event_id)
'''

client.command(DDL_LMS_EVENTS)
client.command('TRUNCATE TABLE lect_04_lms_events')
client.insert_df('lect_04_lms_events', df_events)

rows = client.command('SELECT count() FROM lect_04_lms_events')
print(f'✔ Загружено строк: {rows}')

✔ Загружено строк: 40


In [51]:
client.query_df('''
SELECT *
FROM lect_04_lms_events
ORDER BY student_id, event_time
LIMIT 10
''')

,event_id,student_id,course_id,module_id,event_type,event_time,score,time_spent_min,registration_date,birth_date
0,1,101,CH101,M1,registered,2025-09-01 05:00:00-04:00,NaN,0,2025-09-01,2004-05-20
1,2,101,CH101,M1,first_login,2025-09-01 06:15:00-04:00,NaN,20,2025-09-01,2004-05-20
2,3,101,CH101,M1,module_completed,2025-09-03 14:30:00-04:00,78.0,85,2025-09-01,2004-05-20
3,4,101,CH101,M2,module_completed,2025-09-05 16:10:00-04:00,82.0,90,2025-09-01,2004-05-20
4,5,101,CH101,M3,module_completed,2025-09-10 17:00:00-04:00,88.0,110,2025-09-01,2004-05-20
5,6,101,CH101,FINAL,final_test_passed,2025-09-15 15:30:00-04:00,91.0,70,2025-09-01,2004-05-20
6,7,101,CH101,CERT,certificate_issued,2025-09-16 05:00:00-04:00,NaN,0,2025-09-01,2004-05-20
7,8,102,CH101,M1,registered,2025-09-02 08:00:00-04:00,NaN,0,2025-09-02,2003-11-03
8,9,102,CH101,M1,first_login,2025-09-04 04:20:00-04:00,NaN,15,2025-09-02,2003-11-03
9,10,102,CH101,M1,module_completed,2025-09-06 13:00:00-04:00,64.0,60,2025-09-02,2003-11-03


# Блок A. Функции работы с датой и временем

В LMS почти каждое событие имеет временную метку.  
Наша задача — превратить `DateTime` в аналитические признаки: год, месяц, день недели, час.

```mermaid
flowchart LR
    DT["event_time"] --> Y["toYear"]
    DT --> M["toMonth"]
    DT --> DW["toDayOfWeek"]
    DT --> H["toHour"]
    Y --> A["когорты"]
    M --> B["сезонность"]
    DW --> C["недельный паттерн"]
    H --> D["суточный профиль"]
```

In [52]:
client.query_df('''
SELECT
    event_id,
    student_id,
    event_type,
    event_time,
    toYear(event_time) AS year,
    toMonth(event_time) AS month,
    toDayOfWeek(event_time) AS day_of_week,
    toHour(event_time) AS hour
FROM lect_04_lms_events
ORDER BY event_time
LIMIT 12
''')

,event_id,student_id,event_type,event_time,year,month,day_of_week,hour
0,1,101,registered,2025-09-01 05:00:00-04:00,2025,9,1,5
1,2,101,first_login,2025-09-01 06:15:00-04:00,2025,9,1,6
2,8,102,registered,2025-09-02 08:00:00-04:00,2025,9,2,8
3,13,103,registered,2025-09-03 11:00:00-04:00,2025,9,3,11
4,3,101,module_completed,2025-09-03 14:30:00-04:00,2025,9,3,14
5,9,102,first_login,2025-09-04 04:20:00-04:00,2025,9,4,4
6,17,104,registered,2025-09-04 16:00:00-04:00,2025,9,4,16
7,18,104,first_login,2025-09-04 18:10:00-04:00,2025,9,4,18
8,24,105,registered,2025-09-05 05:10:00-04:00,2025,9,5,5
9,25,105,first_login,2025-09-05 05:40:00-04:00,2025,9,5,5


## A1. Суточный профиль активности

**Вопрос:** в какие часы студенты чаще всего работают с LMS?

In [53]:
client.query_df('''
SELECT
    toHour(event_time) AS hour,
    count() AS events,
    uniq(student_id) AS active_students
FROM lect_04_lms_events
WHERE event_type != 'registered'
GROUP BY hour
ORDER BY hour
''')

,hour,events,active_students
0,4,2,2
1,5,3,3
2,6,2,2
3,7,3,2
4,8,4,3
5,9,1,1
6,10,1,1
7,11,1,1
8,12,1,1
9,13,1,1


### 🧩 Задание A1

Посчитайте количество событий по дням недели.

**Подсказка:** используйте `toDayOfWeek(event_time)` и `GROUP BY`.

In [54]:
# Ваш SQL:
client.query_df('''
SELECT
    toDayOfWeek(event_time) AS day_of_week,
    count() AS events
FROM lect_04_lms_events
GROUP BY day_of_week
ORDER BY day_of_week
''')

,day_of_week,events
0,1,11
1,2,6
2,3,5
3,4,6
4,5,5
5,6,5
6,7,2


# Блок B. Округление даты и времени

Функции `toStartOf...` нужны, когда мы строим временной ряд.

```mermaid
flowchart LR
    E["сырые события LMS"] --> R["toStartOfDay / toStartOfMonth"]
    R --> G["GROUP BY period"]
    G --> TS["DAU / MAU / динамика сдач"]
```

In [55]:
client.query_df('''
SELECT
    toStartOfDay(event_time) AS day,
    count() AS events,
    uniq(student_id) AS dau
FROM lect_04_lms_events
WHERE event_type != 'registered'
GROUP BY day
ORDER BY day
''')

,day,events,dau
0,2025-09-01 00:00:00-04:00,1,1
1,2025-09-03 00:00:00-04:00,1,1
2,2025-09-04 00:00:00-04:00,2,2
3,2025-09-05 00:00:00-04:00,2,2
4,2025-09-06 00:00:00-04:00,2,2
5,2025-09-07 00:00:00-04:00,1,1
6,2025-09-08 00:00:00-04:00,4,2
7,2025-09-09 00:00:00-04:00,3,2
8,2025-09-10 00:00:00-04:00,2,2
9,2025-09-11 00:00:00-04:00,2,2


## B1. DAU: Daily Active Users

`DAU` — количество уникальных активных студентов за день.

Для быстрой аналитики используем `uniq(student_id)`.  
Для строгой отчётности можно заменить на `uniqExact(student_id)`.

In [56]:
dau_df = client.query_df('''
SELECT
    toStartOfDay(event_time) AS day,
    uniq(student_id) AS dau
FROM lect_04_lms_events
WHERE event_type NOT IN ('registered')
GROUP BY day
ORDER BY day
''')
dau_df

,day,dau
0,2025-09-01 00:00:00-04:00,1
1,2025-09-03 00:00:00-04:00,1
2,2025-09-04 00:00:00-04:00,2
3,2025-09-05 00:00:00-04:00,2
4,2025-09-06 00:00:00-04:00,2
5,2025-09-07 00:00:00-04:00,1
6,2025-09-08 00:00:00-04:00,2
7,2025-09-09 00:00:00-04:00,2
8,2025-09-10 00:00:00-04:00,2
9,2025-09-11 00:00:00-04:00,2


In [57]:
import plotly.express as px

fig = px.line(dau_df, x='day', y='dau', markers=True, title='DAU: активные студенты по дням')
fig.update_layout(xaxis_title='День', yaxis_title='Уникальные студенты')
fig.show()

### 🧩 Задание B1

Постройте `MAU` — число уникальных активных студентов по месяцам.

**Подсказка:** используйте `toStartOfMonth(event_time)`.

In [58]:
# Ваш SQL:
client.query_df('''
SELECT
    toStartOfMonth(event_time) AS month,
    uniq(student_id) AS mau
FROM lect_04_lms_events
WHERE event_type != 'registered'
GROUP BY month
ORDER BY month
''')

,month,mau
0,2025-09-01,7


# Блок C. `dateDiff`: интервалы между событиями

`dateDiff` возвращает количество пересечённых границ указанного интервала.  
Это важно: функция считает не «полные периоды», а именно календарные границы.

```mermaid
flowchart LR
    A["registration_date"] --> F["dateDiff('day', registration_date, first_login)"]
    B["first_login"] --> F
    F --> C["days_to_activation"]
```

## C1. Activation: время от регистрации до первого входа

**Метрика:** сколько дней прошло между регистрацией и первым входом.

Сначала найдём для каждого студента дату первого входа через `minIf`.

In [59]:
client.query_df('''
SELECT
    student_id,
    min(registration_date) AS reg_date,
    minIf(event_time, event_type = 'first_login') AS first_login,
    dateDiff('day', reg_date, first_login) AS days_to_activation
FROM lect_04_lms_events
GROUP BY student_id
ORDER BY student_id
''')

,student_id,reg_date,first_login,days_to_activation
0,101,2025-09-01,2025-09-01 06:15:00-04:00,0
1,102,2025-09-02,2025-09-04 04:20:00-04:00,2
2,103,2025-09-03,2025-09-10 11:30:00-04:00,7
3,104,2025-09-04,2025-09-04 18:10:00-04:00,0
4,105,2025-09-05,2025-09-05 05:40:00-04:00,0
5,106,2025-09-07,1969-12-31 19:00:00-05:00,45197
6,107,2025-09-08,2025-09-08 04:05:00-04:00,0
7,108,2025-09-10,2025-09-11 09:20:00-04:00,1


## C2. Время до финального теста

Теперь считаем путь до `final_test_passed`.

In [60]:
client.query_df('''
SELECT
    student_id,
    min(registration_date) AS reg_date,
    minIf(event_time, event_type = 'first_login') AS first_login,
    dateDiff('day', reg_date, first_login) AS days_to_activation
FROM lect_04_lms_events
GROUP BY student_id
ORDER BY student_id
''')

,student_id,reg_date,first_login,days_to_activation
0,101,2025-09-01,2025-09-01 06:15:00-04:00,0
1,102,2025-09-02,2025-09-04 04:20:00-04:00,2
2,103,2025-09-03,2025-09-10 11:30:00-04:00,7
3,104,2025-09-04,2025-09-04 18:10:00-04:00,0
4,105,2025-09-05,2025-09-05 05:40:00-04:00,0
5,106,2025-09-07,1969-12-31 19:00:00-05:00,45197
6,107,2025-09-08,2025-09-08 04:05:00-04:00,0
7,108,2025-09-10,2025-09-11 09:20:00-04:00,1


### 🧩 Задание C1

Посчитайте возраст студента на момент регистрации.

**Подсказка:** используйте `dateDiff('year', birth_date, registration_date)`.  
**Вопрос для обсуждения:** почему такой возраст может отличаться от «полного возраста»?

In [61]:
# Ваш SQL:
client.query_df('''
SELECT
    student_id,
    min(birth_date) AS b_date,
    min(registration_date) AS reg_date,
    dateDiff('year', min(birth_date), min(registration_date)) AS age_at_registration
FROM lect_04_lms_events
GROUP BY student_id
ORDER BY student_id
''')

,student_id,b_date,reg_date,age_at_registration
0,101,2004-05-20,2025-09-01,21
1,102,2003-11-03,2025-09-02,22
2,103,2005-02-17,2025-09-03,20
3,104,1999-07-29,2025-09-04,26
4,105,2002-12-31,2025-09-05,23
5,106,2001-01-15,2025-09-07,24
6,107,2004-09-10,2025-09-08,21
7,108,1998-04-02,2025-09-10,27


# Блок D. Агрегатные функции

Агрегации отвечают на вопрос: что происходит **в группе**?

```mermaid
flowchart TD
    Rows["строки событий"] --> Group["GROUP BY student_id"]
    Group --> Agg["агрегатная функция"]
    Agg --> Result["одна строка на студента"]
```

## D1. `argMax` и `argMin`: лучший и слабый модуль

`argMax(module_id, score)` возвращает модуль, где балл максимальный.  
`argMin(module_id, score)` возвращает модуль, где балл минимальный.

In [62]:
client.query_df('''
SELECT
    student_id,
    argMax(module_id, score) AS best_module,
    max(score) AS best_score,
    argMin(module_id, score) AS weakest_module,
    min(score) AS weakest_score
FROM lect_04_lms_events
WHERE score IS NOT NULL
GROUP BY student_id
ORDER BY student_id
''')

,student_id,best_module,best_score,weakest_module,weakest_score
0,101,FINAL,91.0,M1,78.0
1,102,M2,71.0,M1,64.0
2,104,FINAL,97.0,M1,90.0
3,105,M2,58.0,M1,55.0
4,107,FINAL,89.0,M1,86.0
5,108,M2,81.0,M1,77.0


## D2. `uniq` и `uniqExact`: скорость против точности

In [63]:
client.query_df('''
SELECT
    uniq(student_id) AS approx_students,
    uniqExact(student_id) AS exact_students,
    uniqIf(student_id, event_type = 'final_test_passed') AS approx_finished,
    uniqExactIf(student_id, event_type = 'final_test_passed') AS exact_finished
FROM lect_04_lms_events
''')

,approx_students,exact_students,approx_finished,exact_finished
0,8,8,3,3


## D3. Комбинатор `-If`

Комбинатор `-If` позволяет считать несколько метрик в одном проходе по таблице.

In [64]:
client.query_df('''
SELECT
    uniqExact(student_id) AS total_students,
    uniqExactIf(student_id, event_type = 'first_login') AS activated_students,
    uniqExactIf(student_id, event_type = 'final_test_passed') AS finished_students,
    uniqExactIf(student_id, event_type = 'certificate_issued') AS certified_students,
    countIf(event_type = 'warning') AS warnings,
    countIf(event_type = 'dropout') AS dropouts,
    round(avgIf(score, event_type = 'final_test_passed'), 2) AS avg_final_score
FROM lect_04_lms_events
''')

,total_students,activated_students,finished_students,certified_students,warnings,dropouts,avg_final_score
0,8,7,3,1,2,1,92.33


### 🧩 Задание D1

Посчитайте по каждому студенту:

1. общее время обучения `sum(time_spent_min)`;
2. средний балл `avg(score)`;
3. число завершённых модулей `countIf(event_type = 'module_completed')`.

Отсортируйте результат по убыванию общего времени обучения.

In [65]:
# Ваш SQL:
client.query_df('''
SELECT
    student_id,
    sum(time_spent_min) AS total_time_spent,
    avg(score) AS average_score,
    countIf(event_type = 'module_completed') AS completed_modules
FROM lect_04_lms_events
GROUP BY student_id
ORDER BY total_time_spent DESC
''')

,student_id,total_time_spent,average_score,completed_modules
0,104,475,93.75,3
1,101,375,84.75,3
2,105,255,56.50,2
3,108,255,79.00,2
4,102,230,68.00,3
5,107,220,87.50,3
6,103,5,NaN,0
7,106,0,NaN,0


# Блок E. `groupArray`: траектория студента

`groupArray` собирает значения группы в массив.  
Для образовательной аналитики это способ получить **траекторию обучения**.

```mermaid
flowchart LR
    A["события студента"] --> B["groupArray(event_type)"]
    B --> C["['registered','first_login','module_completed', ...]"]
```

In [66]:
client.query_df('''
SELECT
    student_id,
    groupArray(event_type) AS events,
    groupArray(module_id) AS modules
FROM (
    SELECT *
    FROM lect_04_lms_events
    ORDER BY student_id, event_time, event_id
)
GROUP BY student_id
ORDER BY student_id
''')

,student_id,events,modules
0,101,"[registered, first_login, module_completed, module_completed, module_completed, final_test_passed, certificate_issued]","[M1, M1, M1, M2, M3, FINAL, CERT]"
1,102,"[registered, first_login, module_completed, module_completed, module_completed]","[M1, M1, M1, M2, M3]"
2,103,"[registered, first_login, warning, dropout]","[M1, M1, M1, M1]"
3,104,"[registered, first_login, module_completed, module_completed, module_completed, advanced_started, final_test_passed]","[M1, M1, M1, M2, M3, ADV1, FINAL]"
4,105,"[registered, first_login, module_completed, module_completed, warning]","[M1, M1, M1, M2, M2]"
5,106,[registered],[M1]
6,107,"[registered, first_login, module_completed, module_completed, module_completed, final_test_passed]","[M1, M1, M1, M2, M3, FINAL]"
7,108,"[registered, first_login, module_completed, module_completed, advanced_started]","[M1, M1, M1, M2, ADV2]"


## E1. `indexOf`: на каком шаге студент сдал финальный тест?

Если событие не найдено, `indexOf` вернёт `0`.

In [67]:
client.query_df('''
WITH trajectories AS (
    SELECT
        student_id,
        groupArray(event_type) AS events
    FROM (
        SELECT *
        FROM lect_04_lms_events
        ORDER BY student_id, event_time, event_id
    )
    GROUP BY student_id
)
SELECT
    student_id,
    events,
    indexOf(events, 'final_test_passed') AS final_test_step
FROM trajectories
ORDER BY student_id
''')

,student_id,events,final_test_step
0,101,"[registered, first_login, module_completed, module_completed, module_completed, final_test_passed, certificate_issued]",6
1,102,"[registered, first_login, module_completed, module_completed, module_completed]",0
2,103,"[registered, first_login, warning, dropout]",0
3,104,"[registered, first_login, module_completed, module_completed, module_completed, advanced_started, final_test_passed]",7
4,105,"[registered, first_login, module_completed, module_completed, warning]",0
5,106,[registered],0
6,107,"[registered, first_login, module_completed, module_completed, module_completed, final_test_passed]",6
7,108,"[registered, first_login, module_completed, module_completed, advanced_started]",0


## E2. `arraySlice`: первые шаги студента

Посмотрим первые 3 события в траектории.

In [68]:
client.query_df('''
WITH trajectories AS (
    SELECT
        student_id,
        groupArray(event_type) AS events
    FROM (
        SELECT *
        FROM lect_04_lms_events
        ORDER BY student_id, event_time, event_id
    )
    GROUP BY student_id
)
SELECT
    student_id,
    arraySlice(events, 1, 3) AS first_three_events,
    arraySlice(events, -3) AS last_three_events
FROM trajectories
ORDER BY student_id
''')

,student_id,first_three_events,last_three_events
0,101,"[registered, first_login, module_completed]","[module_completed, final_test_passed, certificate_issued]"
1,102,"[registered, first_login, module_completed]","[module_completed, module_completed, module_completed]"
2,103,"[registered, first_login, warning]","[first_login, warning, dropout]"
3,104,"[registered, first_login, module_completed]","[module_completed, advanced_started, final_test_passed]"
4,105,"[registered, first_login, module_completed]","[module_completed, module_completed, warning]"
5,106,[registered],[registered]
6,107,"[registered, first_login, module_completed]","[module_completed, module_completed, final_test_passed]"
7,108,"[registered, first_login, module_completed]","[module_completed, module_completed, advanced_started]"


### 🧩 Задание E1

Для каждого студента получите:

1. массив `modules`;
2. первый модуль `modules[1]`;
3. последние два модуля через `arraySlice(modules, -2)`.

**Подсказка:** сначала отсортируйте события во внутреннем запросе.

In [69]:
# Ваш SQL:
client.query_df('''
WITH trajectories AS (
    SELECT
        student_id,
        groupArray(module_id) AS modules
    FROM (
        SELECT * FROM lect_04_lms_events
        ORDER BY student_id, event_time, event_id
    )
    GROUP BY student_id
)
SELECT
    student_id,
    modules,
    modules[1] AS first_module,
    arraySlice(modules, -2) AS last_two_modules
FROM trajectories
ORDER BY student_id
''')

,student_id,modules,first_module,last_two_modules
0,101,"[M1, M1, M1, M2, M3, FINAL, CERT]",M1,"[FINAL, CERT]"
1,102,"[M1, M1, M1, M2, M3]",M1,"[M2, M3]"
2,103,"[M1, M1, M1, M1]",M1,"[M1, M1]"
3,104,"[M1, M1, M1, M2, M3, ADV1, FINAL]",M1,"[ADV1, FINAL]"
4,105,"[M1, M1, M1, M2, M2]",M1,"[M2, M2]"
5,106,[M1],M1,[M1]
6,107,"[M1, M1, M1, M2, M3, FINAL]",M1,"[M3, FINAL]"
7,108,"[M1, M1, M1, M2, ADV2]",M1,"[M2, ADV2]"


# Блок F. Поиск элементов в массивах: `has`, `hasAll`, `hasAny`

Эти функции возвращают `1` или `0`.

| Функция | Вопрос |
|---|---|
| `has(array, x)` | есть ли один элемент? |
| `hasAll(array, [x, y])` | есть ли все элементы? |
| `hasAny(array, [x, y])` | есть ли хотя бы один элемент? |

In [70]:
client.query_df('''
WITH trajectories AS (
    SELECT
        student_id,
        groupArray(module_id) AS modules,
        groupArray(event_type) AS events
    FROM (
        SELECT *
        FROM lect_04_lms_events
        ORDER BY student_id, event_time, event_id
    )
    GROUP BY student_id
)
SELECT
    student_id,
    modules,
    hasAll(modules, ['M1', 'M2', 'M3']) AS finished_core_modules,
    hasAny(modules, ['ADV1', 'ADV2']) AS touched_advanced_track,
    has(events, 'certificate_issued') AS received_certificate,
    hasAny(events, ['warning', 'dropout']) AS risk_signal
FROM trajectories
ORDER BY student_id
''')

,student_id,modules,finished_core_modules,touched_advanced_track,received_certificate,risk_signal
0,101,"[M1, M1, M1, M2, M3, FINAL, CERT]",1,0,1,0
1,102,"[M1, M1, M1, M2, M3]",1,0,0,0
2,103,"[M1, M1, M1, M1]",0,0,0,1
3,104,"[M1, M1, M1, M2, M3, ADV1, FINAL]",1,1,0,0
4,105,"[M1, M1, M1, M2, M2]",0,0,0,1
5,106,[M1],0,0,0,0
6,107,"[M1, M1, M1, M2, M3, FINAL]",1,0,0,0
7,108,"[M1, M1, M1, M2, ADV2]",0,1,0,0


## F1. Сегментация студентов

На основе массивов можно получить сегменты:

- `completed_core` — прошёл M1, M2, M3;
- `advanced` — начал ADV1 или ADV2;
- `risk` — были warning или dropout;
- `certified` — получил сертификат.

In [71]:
client.query_df('''
WITH trajectories AS (
    SELECT
        student_id,
        groupArray(module_id) AS modules,
        groupArray(event_type) AS events
    FROM (
        SELECT *
        FROM lect_04_lms_events
        ORDER BY student_id, event_time, event_id
    )
    GROUP BY student_id
),
segments AS (
    SELECT
        student_id,
        hasAll(modules, ['M1', 'M2', 'M3']) AS completed_core,
        hasAny(modules, ['ADV1', 'ADV2']) AS advanced,
        hasAny(events, ['warning', 'dropout']) AS risk,
        has(events, 'certificate_issued') AS certified
    FROM trajectories
)
SELECT
    count() AS students,
    sum(completed_core) AS completed_core_students,
    sum(advanced) AS advanced_students,
    sum(risk) AS risk_students,
    sum(certified) AS certified_students,
    round(avg(completed_core) * 100, 1) AS completed_core_pct,
    round(avg(risk) * 100, 1) AS risk_pct
FROM segments
''')

,students,completed_core_students,advanced_students,risk_students,certified_students,completed_core_pct,risk_pct
0,8,4,2,2,1,50.0,25.0


### 🧩 Задание F1

Найдите студентов, которые:

1. прошли `M1` и `M2`;
2. не прошли `M3`;
3. имеют хотя бы один сигнал риска (`warning` или `dropout`).

Это потенциальная группа для преподавательского вмешательства.

In [72]:
# Ваш SQL:
client.query_df('''
WITH trajectories AS (
    SELECT
        student_id,
        groupArray(module_id) AS modules,
        groupArray(event_type) AS events
    FROM (SELECT * FROM lect_04_lms_events ORDER BY event_time)
    GROUP BY student_id
)
SELECT student_id
FROM trajectories
WHERE
    hasAll(modules,['M1', 'M2']) = 1
    AND has(modules, 'M3') = 0
    AND hasAny(events, ['warning', 'dropout']) = 1
''')

,student_id
0,105


# Итоговый кейс: мини-дашборд курса

Соберём ключевые показатели курса в одном запросе:

- всего студентов;
- активированных студентов;
- завершивших финальный тест;
- получивших сертификат;
- студентов с сигналами риска;
- средний финальный балл;
- среднее время до активации.

In [73]:
client.query_df('''
WITH student_level AS (
    SELECT
        student_id,
        min(registration_date) AS registration_date,
        minIf(event_time, event_type = 'first_login') AS first_login_time,
        minIf(event_time, event_type = 'final_test_passed') AS final_test_time,
        countIf(event_type = 'certificate_issued') > 0 AS certified,
        countIf(event_type IN ('warning', 'dropout')) > 0 AS risk,
        maxIf(score, event_type = 'final_test_passed') AS final_score,
        sum(time_spent_min) AS total_time_spent_min
    FROM lect_04_lms_events
    GROUP BY student_id
)
SELECT
    count() AS total_students,
    countIf(first_login_time IS NOT NULL) AS activated_students,
    countIf(final_test_time IS NOT NULL) AS finished_students,
    sum(certified) AS certified_students,
    sum(risk) AS risk_students,
    round(avgIf(final_score, final_score IS NOT NULL), 2) AS avg_final_score,
    round(avgIf(dateDiff('day', registration_date, first_login_time), first_login_time IS NOT NULL), 2) AS avg_days_to_activation,
    round(avg(total_time_spent_min), 1) AS avg_time_spent_min
FROM student_level
''')

,total_students,activated_students,finished_students,certified_students,risk_students,avg_final_score,avg_days_to_activation,avg_time_spent_min
0,8,8,8,1,2,92.33,5650.88,226.9


# Контрольная работа

Выполните задания ниже в отдельных SQL-ячейках.

## Задание 1. Недельный профиль
Постройте распределение активных студентов по дням недели.

## Задание 2. Доля завершения
Посчитайте долю студентов, которые дошли до `final_test_passed`.

## Задание 3. Траектория риска
Найдите траектории студентов, где встречается `warning` или `dropout`.

## Задание 4. Скорость прохождения
Посчитайте, за сколько дней студенты доходят от регистрации до финального теста.

## Задание 5. Продвинутый трек
Определите, кто начал `ADV1` или `ADV2`, и сравните их средний балл с остальными.

In [ ]:
# Контрольная работа: пишите решения здесь

In [74]:
# 1. Постройте распределение активных студентов по дням недели.
client.query_df('''
SELECT
    toDayOfWeek(event_time) AS week_day,
    uniq(student_id) AS wau
FROM lect_04_lms_events
WHERE event_type != 'registered'
GROUP BY week_day
ORDER BY week_day
''')

,week_day,wau
0,1,6
1,2,4
2,3,2
3,4,3
4,5,3
5,6,4
6,7,1


In [76]:
# 2. Посчитайте долю студентов, которые дошли до final_test_passed.
client.query_df('''
SELECT
    uniq(student_id) AS total_students,
    uniqIf(student_id, event_type = 'final_test_passed') AS finished_students,
    round((finished_students / total_students) * 100, 1) AS completion_rate
FROM lect_04_lms_events
''')

,total_students,finished_students,completion_rate
0,8,3,37.5


In [86]:
# 3. Найдите траектории студентов, где встречается warning или dropout.
client.query_df('''
WITH students_tracks AS (
    SELECT
        student_id,
        groupArray(event_type) AS track
    FROM (
        SELECT *
        FROM lect_04_lms_events
        ORDER BY event_time
    )
    GROUP BY student_id
)
SELECT
    student_id,
    track
FROM students_tracks
WHERE hasAny(track, ['warning', 'dropout'])
''')

,student_id,track
0,105,"[registered, first_login, module_completed, module_completed, warning]"
1,103,"[registered, first_login, warning, dropout]"


In [89]:
# 4. Посчитайте, среднее количество дней, за которое студенты доходят от регистрации до финального теста.
client.query_df('''
WITH student_times AS (
    SELECT
        student_id,
        min(registration_date) AS reg_date,
        minIf(event_time, event_type = 'final_test_passed') AS final_time
    FROM lect_04_lms_events
    GROUP BY student_id
    HAVING final_time IS NOT NULL
)
SELECT
    round(avg(dateDiff('day', reg_date, final_time)), 1) AS avg_days_to_complete
FROM student_times
''')


,avg_days_to_complete
0,28252.8


In [78]:
# 5. Определите, кто начал ADV1 или ADV2, и сравните их средний балл с остальными.
client.query_df('''
WITH trajectories AS (
    SELECT
        student_id,
        groupArray(module_id) AS modules,
        maxIf(score, event_type = 'final_test_passed') AS final_score
    FROM (
        SELECT *
        FROM lect_04_lms_events
        ORDER BY student_id, event_time, event_id
    )
    GROUP BY student_id
),
segments AS (
    SELECT
        student_id,
        final_score,
        hasAny(modules, ['ADV1', 'ADV2']) AS advanced_track
    FROM trajectories
    WHERE final_score IS NOT NULL
)
SELECT
    CASE
        WHEN advanced_track = 1 THEN 'Advanced track'
        ELSE 'Standard track'
    END AS cohort,
    count() AS students_count,
    round(avg(final_score), 2) AS avg_final_score,
    min(final_score) AS min_score,
    max(final_score) AS max_score
FROM segments
GROUP BY advanced_track
ORDER BY avg_final_score DESC
''')

,cohort,students_count,avg_final_score,min_score,max_score
0,Advanced track,1,97.0,97.0,97.0
1,Standard track,2,90.0,89.0,91.0


# Методические комментарии для преподавателя

## Рекомендуемый сценарий занятия

1. **10 минут** — объяснить сквозной кейс LMS и структуру таблицы.
2. **15 минут** — дата/время и временные ряды: `toHour`, `toStartOfDay`.
3. **15 минут** — интервалы: `dateDiff`, activation, time-to-final.
4. **20 минут** — агрегаты: `argMax`, `uniq`, `uniqExact`, `-If`.
5. **20 минут** — массивы: `groupArray`, `indexOf`, `arraySlice`, `has*`.
6. **20 минут** — самостоятельная работа и разбор контрольных заданий.

## Ключевые акценты

- ClickHouse особенно силён в событийной аналитике.
- В образовательной аналитике почти все метрики строятся из событий.
- `toStartOf...` нужен для временных рядов, а `to...` — для признаков.
- `dateDiff` считает границы интервала, поэтому возраст и длительность требуют аккуратной интерпретации.
- `groupArray` превращает события в траекторию студента.
- `has/hasAll/hasAny` позволяют быстро сегментировать студентов по прогрессу.